# Competitive CeNN — theoretical feasibility before SmolLM2-360M replacement

**Goal:** Test a new standalone cellular-memory candidate *before* replacing another pretrained attention layer. First run CPU mathematical and synthetic learning tests; then on a Tesla T4 / other CUDA GPU inspect the **original SmolLM2-360M-Instruct layer-0 attention distributions**. No pretrained layer is replaced by this notebook.

**Critical scope:** content-dependent cell routing is functionally **attention-like** although the candidate has no Transformer attention block, no QKV attention matrix, no SDPA and no DeltaNet/LSTM. Novelty and speedups are **not established**.

**Experiment design:** Memory scales with cell count; local nearest-neighbor winners travel up a binary tree. If native attention is diffuse, hard selection cannot preserve it. This notebook checks that hypothesis rather than publishing another failed checkpoint.
**Setup:** The research implementation currently lives on the `research/competitive-cenn-feasibility` branch, not on `main`. The setup cell uses its own checkout directory and verifies the module import; it is safe to run while another `/content/TinyCeNN-LM` checkout exists.


In [1]:
#@title 1. Prepare environment (isolated research checkout + import smoke test)
import os, sys, subprocess, json, pathlib, importlib, torch
print('PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name())

REPO_URL = 'https://github.com/vtavakkoli/TinyCeNN-LM.git'
BRANCH = 'research/competitive-cenn-feasibility'
# Do NOT reuse /content/TinyCeNN-LM: it may already contain main, which has no CCM module.
workspace = pathlib.Path('/content') if pathlib.Path('/content').is_dir() else pathlib.Path.cwd()
root = workspace / 'TinyCeNN-LM-competitive-feasibility'
if root.exists():
    if not (root / '.git').is_dir():
        raise RuntimeError(f'{root} exists but is not a Git checkout. Move or remove it first.')
    origin = subprocess.check_output(
        ['git', '-C', str(root), 'remote', 'get-url', 'origin'], text=True
    ).strip()
    if origin.removesuffix('.git') != REPO_URL.removesuffix('.git'):
        raise RuntimeError(f'Unexpected repository remote: {origin}')
    subprocess.check_call(['git', '-C', str(root), 'fetch', '--depth', '1', 'origin', BRANCH])
    subprocess.check_call(['git', '-C', str(root), 'checkout', '--detach', 'FETCH_HEAD'])
else:
    subprocess.check_call([
        'git', 'clone', '--depth', '1', '--single-branch', '--branch', BRANCH, REPO_URL, str(root)
    ])

required = [
    'src/tinycenn_lm/competitive_cenn.py',
    'src/tinycenn_lm/competitive_cenn_experiment.py',
    'scripts/attention_concentration_smol360.py',
    'tests/test_competitive_cenn.py',
]
missing = [p for p in required if not (root / p).is_file()]
if missing:
    raise RuntimeError(f'The {BRANCH} checkout is missing required files: {missing}')

# Pin the attention diagnostic's Transformers API. Editable installs made *inside*
# an already running notebook do not always refresh its import search path.
subprocess.check_call([
    sys.executable, '-m', 'pip', '-q', 'install', 'transformers==4.57.6', 'pytest'
])
subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', '-e', str(root), '--no-deps'])
for search_path in (str(root / 'src'), str(root)):
    if search_path in sys.path:
        sys.path.remove(search_path)
    sys.path.insert(0, search_path)
os.environ['PYTHONPATH'] = os.pathsep.join(
    [str(root / 'src'), str(root), os.environ.get('PYTHONPATH', '')]
)
importlib.invalidate_caches()
# Prevent a previously imported module from a different TinyCeNN checkout
# from masking this branch in a reused Colab session.
old_package = sys.modules.get('tinycenn_lm')
if old_package is not None:
    old_file = pathlib.Path(getattr(old_package, '__file__', '') or '/').resolve()
    if not old_file.is_relative_to((root / 'src').resolve()):
        for name in list(sys.modules):
            if name == 'tinycenn_lm' or name.startswith('tinycenn_lm.'):
                sys.modules.pop(name, None)
os.chdir(root)
from tinycenn_lm.competitive_cenn_experiment import run_symbolic_retrieval, run_diffuse_gap
from scripts.attention_concentration_smol360 import analyze_probability_value
commit = subprocess.check_output(['git', '-C', str(root), 'rev-parse', 'HEAD'], text=True).strip()
print('Research checkout:', root)
print('Branch:', BRANCH, 'commit:', commit)
print('CeNN module:', importlib.import_module('tinycenn_lm.competitive_cenn_experiment').__file__)
print('Import smoke tests: PASS')


PyTorch: 2.11.0+cu130 CUDA: True
Device: Tesla T4
Research checkout: /content/TinyCeNN-LM-competitive-feasibility
Branch: research/competitive-cenn-feasibility commit: c85dab956fb3e154508cf29c7456d5a99ecaa7d3
CeNN module: /content/TinyCeNN-LM-competitive-feasibility/src/tinycenn_lm/competitive_cenn_experiment.py
Import smoke tests: PASS


In [2]:
#@title 2. Run causal and gradient tests (CPU)
subprocess.check_call([sys.executable,'-m','pytest','-q','tests/test_competitive_cenn.py'])

0

In [3]:
#@title 3. New cellular layer vs peaked, content-selective teacher
import importlib.util
from tinycenn_lm.competitive_cenn_experiment import run_symbolic_retrieval, run_diffuse_gap
out_dir=pathlib.Path('/content/ccm_cenn_feasibility' if pathlib.Path('/content').exists() else './ccm_cenn_feasibility')
out_dir.mkdir(exist_ok=True,parents=True)
for slots, ids, steps in [(8,16,400),(16,32,400),(32,64,400)]:
    report=run_symbolic_retrieval(steps=steps,slots=slots,ids=ids)
    (out_dir/f'learned_{slots}.json').write_text(json.dumps(report,indent=2))
    print('Cells',slots,'learned relative error:',report['history'][-1]['relative_error'])
gap=run_diffuse_gap()
(out_dir/'softmax_gap.json').write_text(json.dumps(gap,indent=2))
print('Diffuse attention approximation gaps (attention teacher only):')
for r in gap:
    if r['slots']==16 and r['temperature_multiplier'] in (1,4):
        print(r)

Cells 8 learned relative error: 0.004630318462466794
Cells 16 learned relative error: 0.000978021346430901
Cells 32 learned relative error: 0.002042888701142079
Diffuse attention approximation gaps (attention teacher only):
{'slots': 16, 'temperature_multiplier': 1, 'average_max_probability': 0.24296677112579346, 'hard_routing_mse': 0.6447042226791382, 'hard_routing_relative_mse': 4.908339454723532}
{'slots': 16, 'temperature_multiplier': 4, 'average_max_probability': 0.6946144104003906, 'hard_routing_mse': 0.19563928246498108, 'hard_routing_relative_mse': 0.3369485457532376}


In [4]:
#@title 4. Probe native first attention layer (GPU required; no student model trained)
if not torch.cuda.is_available():
    print('GPU unavailable: cannot claim real SmolLM2 attention-layer feasibility. Please choose a Colab GPU runtime.')
else:
    subprocess.check_call([sys.executable,'scripts/attention_concentration_smol360.py',
                           '--device','cuda','--lengths','128','256',
                           '--output',str(out_dir/'smol360_attention_diagnostic.json')])
    native=json.loads((out_dir/'smol360_attention_diagnostic.json').read_text())
    for x in native['results']:
        print('seq=',x['length'],'case=',x['text_case'],
              'top1mass=',round(x['topk']['1']['mean_probability_mass'],3),
              'top4mass=',round(x['topk']['4']['mean_probability_mass'],3),
              'top4 oracle MSE=',round(x['topk']['4']['output_relative_mse'],3))
    # A diagnostic criterion only, NOT an acceptance gate for deployment.
    go=all(x['topk']['4']['mean_probability_mass']>0.90 and
           x['topk']['4']['output_relative_mse']<0.05 for x in native['results'])
    print('Top-4 selective-cell feasibility screen:', 'PROMISING' if go else 'NOT PROVEN')

seq= 128 case= 0 top1mass= 0.098 top4mass= 0.229 top4 oracle MSE= 0.271
seq= 128 case= 1 top1mass= 0.114 top4mass= 0.248 top4 oracle MSE= 0.209
seq= 128 case= 2 top1mass= 0.099 top4mass= 0.225 top4 oracle MSE= 0.22
seq= 256 case= 0 top1mass= 0.07 top4mass= 0.161 top4 oracle MSE= 0.34
seq= 256 case= 1 top1mass= 0.085 top4mass= 0.181 top4 oracle MSE= 0.254
seq= 256 case= 2 top1mass= 0.071 top4mass= 0.157 top4 oracle MSE= 0.269
Top-4 selective-cell feasibility screen: NOT PROVEN


In [5]:
#@title 5. Package reproducible results into a ZIP (even for negative outcomes)
import zipfile
archive=out_dir.parent/(out_dir.name+'_results.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for f in sorted(out_dir.rglob('*')):
        if f.is_file():z.write(f,f.relative_to(out_dir))
print('Saved:',archive)
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    print('Non-Colab: ZIP is available at the printed path.')

Saved: /content/ccm_cenn_feasibility_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## How to interpret the results

A successful symbolic-identifier lookup at 8–32 cells is not proof of continuous-key softmax approximation. The native 360M attention diagnostic is decisive for deciding whether top-k competitive cell routing is even *plausible*. If the first-layer heads have high entropy, a hard-winner cellular architecture loses important weighted mixtures: do not begin another expensive teacher-distillation run. Examine head-by-head dispersion and full-model held-out NLL in a separate study. No GPU performance claim is made here.